In [22]:
import os
import time
import requests
import pandas as pd
from datetime import datetime

# ============================================================================
# RUTAS - MODIFICA SEGÚN TU ESTRUCTURA
# ============================================================================
BASE_DIR = r"C:\Users\josit\CUARTO CURSO\TFG\TFG_Crypto_DEF"

# CSV principal (entrada/salida)
RUTA_DF_MERGED = os.path.join(BASE_DIR, "data", "csv", "df_merged.csv")

# CSV de market cap total histórico (el que descargaste de CoinGecko web)
RUTA_TOTAL_MCAP = os.path.join(BASE_DIR, "data", "csv", "raw", "CoinGecko-GlobalCryptoMktCap-2026-05-02.csv")

# Opcionales (para futuro)
# RUTA_INFLATION = os.path.join(BASE_DIR, "data", "csv", "raw", "inflation.csv")
# RUTA_FED_RATE = os.path.join(BASE_DIR, "data", "csv", "raw", "fed_rate.csv")

# ============================================================================
# APIS
# ============================================================================
COINGECKO_BASE = "https://api.coingecko.com/api/v3"
ALTERNATIVE_BASE = "https://api.alternative.me/fng/"

# Rate limit (free tier: ~10-30 calls/min)
PAUSA_API = 2.5

print("✓ Configuración cargada")
print(f"  df_merged     : {RUTA_DF_MERGED}")
print(f"  Total mcap CSV: {RUTA_TOTAL_MCAP}")

✓ Configuración cargada
  df_merged     : C:\Users\josit\CUARTO CURSO\TFG\TFG_Crypto_DEF\data\csv\df_merged.csv
  Total mcap CSV: C:\Users\josit\CUARTO CURSO\TFG\TFG_Crypto_DEF\data\csv\raw\CoinGecko-GlobalCryptoMktCap-2026-05-02.csv


In [23]:
print("📂 Cargando df_merged.csv...")

df_merged = pd.read_csv(RUTA_DF_MERGED, parse_dates=["date"], index_col="date")
df_merged.sort_index(inplace=True)

ultima_fecha = df_merged.index.max().normalize()
hoy = pd.Timestamp.now().normalize()

print(f"  ✓ Cargado: {len(df_merged)} registros")
print(f"  📅 Última fecha: {ultima_fecha.date()}")
print(f"  📅 Hoy        : {hoy.date()}")
print(f"\n  Columnas ({len(df_merged.columns)}):")
print(f"  {df_merged.columns.tolist()}")

df_merged.tail(3)

📂 Cargando df_merged.csv...
  ✓ Cargado: 2885 registros
  📅 Última fecha: 2025-12-25
  📅 Hoy        : 2026-05-02

  Columnas (19):
  ['btc_open', 'btc_high', 'btc_low', 'btc_close', 'btc_volume', 'eth_open', 'eth_high', 'eth_low', 'eth_close', 'eth_volume', 'btc_dominance', 'eth_dominance', 'alt_dominance', 'fear_greed', 'FearGreed_Label', 'inflation', 'btc_mcap', 'eth_mcap', 'fed_rate']


,btc_open,btc_high,btc_low,btc_close,btc_volume,eth_open,eth_high,eth_low,eth_close,eth_volume,btc_dominance,eth_dominance,alt_dominance,fear_greed,FearGreed_Label,inflation,btc_mcap,eth_mcap,fed_rate
date,,,,,,,,,,,,,,,,,,,
2025-12-23,88490.031250,88898.382812,86606.976562,87414.000000,43683011533,3006.073975,3033.196289,2902.333008,2963.374023,21453213168,0.573204,0.117729,0.309067,24.0,Extreme Fear,2.7,1.767081e+12,3.629357e+11,3.84
2025-12-24,87404.320312,87956.882812,86411.796875,87611.960938,25550297986,2962.922607,2975.171143,2888.988037,2945.590576,13984770325,0.573230,0.117515,0.309256,24.0,Extreme Fear,2.7,1.745209e+12,3.577764e+11,3.84
2025-12-25,87608.320312,88501.789062,86949.257812,87234.742188,19953216347,2945.423096,2968.705078,2891.108887,2903.579590,11828871179,0.574597,0.116787,0.308616,23.0,Extreme Fear,2.7,1.749379e+12,3.555614e+11,3.84


In [24]:
dias_faltantes = (hoy - ultima_fecha).days

print(f"📊 Días a descargar: {dias_faltantes}")

if dias_faltantes <= 0:
    print("⚠️  El CSV ya está actualizado. No hay nada que hacer.")
    raise SystemExit("CSV actualizado")

📊 Días a descargar: 128


In [25]:
def ajustar_dias_coingecko(dias_necesarios):
    """
    CoinGecko free tier solo acepta valores discretos en /market_chart.
    Redondea al siguiente valor válido.
    """
    valores_validos = [1, 7, 14, 30, 90, 180, 365]
    for v in valores_validos:
        if dias_necesarios <= v:
            return v
    return 365

def descargar_market_chart(coin_id, dias):
    """
    Descarga price, volume, market_cap de CoinGecko /market_chart.
    Devuelve DataFrame con ['close', 'volume', 'market_cap'] indexado por fecha.
    """
    url = f"{COINGECKO_BASE}/coins/{coin_id}/market_chart"
    params = {"vs_currency": "usd", "days": dias, "interval": "daily"}
    
    print(f"  → Descargando {coin_id}...")
    r = requests.get(url, params=params, timeout=30)
    
    if r.status_code != 200:
        raise RuntimeError(f"Error {r.status_code}: {r.text[:200]}")
    
    data = r.json()
    
    # Parsear datos
    df_price = pd.DataFrame(data["prices"], columns=["ts", "close"])
    df_vol = pd.DataFrame(data["total_volumes"], columns=["ts", "volume"])
    df_mcap = pd.DataFrame(data["market_caps"], columns=["ts", "market_cap"])
    
    # Convertir timestamps a fechas
    for df in (df_price, df_vol, df_mcap):
        df["date"] = pd.to_datetime(df["ts"], unit="ms").dt.normalize()
        df.drop(columns="ts", inplace=True)
        df.set_index("date", inplace=True)
    
    # Join
    df = df_price.join(df_vol).join(df_mcap)
    
    # Deduplicar (por si hay varios timestamps por día)
    df = df.groupby(df.index).last()
    
    return df

def descargar_fear_greed(limit_dias):
    """
    Descarga Fear & Greed histórico de Alternative.me.
    """
    print(f"  → Descargando Fear & Greed...")
    url = ALTERNATIVE_BASE
    params = {"limit": limit_dias, "format": "json"}
    
    r = requests.get(url, params=params, timeout=30)
    if r.status_code != 200:
        raise RuntimeError(f"Error {r.status_code}")
    
    data = r.json()["data"]
    df = pd.DataFrame(data)
    
    df["date"] = pd.to_datetime(df["timestamp"].astype(int), unit="s").dt.normalize()
    df["fear_greed"] = df["value"].astype(int)
    df["FearGreed_Label"] = df["value_classification"]
    
    return df[["date", "fear_greed", "FearGreed_Label"]].set_index("date").sort_index()

print("✓ Funciones definidas")

✓ Funciones definidas


In [26]:
# Ajustar días al valor válido más cercano
dias_validos = ajustar_dias_coingecko(dias_faltantes + 2)
print(f"🌐 Solicitando {dias_validos} días a CoinGecko (cubre {dias_faltantes} + margen)\n")

# Bitcoin
df_btc = descargar_market_chart("bitcoin", dias_validos)
df_btc.columns = ["btc_close", "btc_volume", "btc_market_cap"]
print(f"  ✓ Bitcoin: {len(df_btc)} registros ({df_btc.index.min().date()} → {df_btc.index.max().date()})")
time.sleep(PAUSA_API)

# Ethereum
df_eth = descargar_market_chart("ethereum", dias_validos)
df_eth.columns = ["eth_close", "eth_volume", "eth_market_cap"]
print(f"  ✓ Ethereum: {len(df_eth)} registros ({df_eth.index.min().date()} → {df_eth.index.max().date()})")
time.sleep(PAUSA_API)

print(f"\n✓ Descarga completada")

🌐 Solicitando 180 días a CoinGecko (cubre 128 + margen)

  → Descargando bitcoin...
  ✓ Bitcoin: 180 registros (2025-11-04 → 2026-05-02)
  → Descargando ethereum...
  ✓ Ethereum: 180 registros (2025-11-04 → 2026-05-02)

✓ Descarga completada


In [27]:
df_feargreed = descargar_fear_greed(limit_dias=dias_validos)
print(f"  ✓ Fear & Greed: {len(df_feargreed)} registros ({df_feargreed.index.min().date()} → {df_feargreed.index.max().date()})")

print(f"\n✓ Fear & Greed descargado")

  → Descargando Fear & Greed...
  ✓ Fear & Greed: 180 registros (2025-11-04 → 2026-05-02)

✓ Fear & Greed descargado


In [28]:
print("📂 Cargando CSV de market cap total histórico...")

# Cargar CSV
df_total_mcap = pd.read_csv(RUTA_TOTAL_MCAP)

# Convertir timestamp de milisegundos a fecha
df_total_mcap["date"] = pd.to_datetime(df_total_mcap["snapped_at"], unit="ms").dt.normalize()
df_total_mcap = df_total_mcap[["date", "market_cap", "total_volume"]]
df_total_mcap.set_index("date", inplace=True)
df_total_mcap.sort_index(inplace=True)

# Renombrar para claridad
df_total_mcap.rename(columns={"market_cap": "total_market_cap"}, inplace=True)

print(f"  ✓ Cargado: {len(df_total_mcap)} registros")
print(f"  📅 Rango: {df_total_mcap.index.min().date()} → {df_total_mcap.index.max().date()}")
print(f"  📊 Último valor: ${df_total_mcap['total_market_cap'].iloc[-1]:,.0f}")

df_total_mcap.tail(3)

📂 Cargando CSV de market cap total histórico...
  ✓ Cargado: 4732 registros
  📅 Rango: 2013-04-29 → 2026-05-02
  📊 Último valor: $2,675,823,384,844


,total_market_cap,total_volume
date,,
2026-04-30,2.615160e+12,1.059284e+11
2026-05-01,2.627642e+12,7.729016e+10
2026-05-02,2.675823e+12,8.559830e+10


In [29]:
print("🔧 Creando OHLC desde close (limitación API gratuita)...")

# Para BTC y ETH: open = high = low = close
for activo in ["btc", "eth"]:
    close_col = f"{activo}_close"
    
    df_btc[f"{activo}_open"] = df_btc[close_col] if activo == "btc" else None
    df_btc[f"{activo}_high"] = df_btc[close_col] if activo == "btc" else None
    df_btc[f"{activo}_low"] = df_btc[close_col] if activo == "btc" else None
    
    if activo == "eth":
        df_eth["eth_open"] = df_eth["eth_close"]
        df_eth["eth_high"] = df_eth["eth_close"]
        df_eth["eth_low"] = df_eth["eth_close"]

# Reordenar columnas: OHLCV
df_btc = df_btc[["btc_open", "btc_high", "btc_low", "btc_close", "btc_volume", "btc_market_cap"]]
df_eth = df_eth[["eth_open", "eth_high", "eth_low", "eth_close", "eth_volume", "eth_market_cap"]]

print("  ✓ OHLC creado")

🔧 Creando OHLC desde close (limitación API gratuita)...
  ✓ OHLC creado


In [30]:
print("🔧 Combinando datos y calculando dominancias...\n")

# 1. Combinar BTC + ETH
df_nuevo = df_btc.join(df_eth, how="outer")
print(f"  ✓ BTC + ETH combinados: {len(df_nuevo)} registros")

# 2. Join con market cap total
df_nuevo = df_nuevo.join(df_total_mcap[["total_market_cap"]], how="left")

# 3. CALCULAR DOMINANCIAS DÍA A DÍA
print(f"\n  🧮 Calculando dominancias día a día...")

dominancias = []

for fecha, row in df_nuevo.iterrows():
    btc_mcap = row["btc_market_cap"]
    eth_mcap = row["eth_market_cap"]
    total_mcap = row["total_market_cap"]
    
    if pd.notna(total_mcap) and total_mcap > 0:
        # Calcular market cap de altcoins
        alt_mcap = total_mcap - btc_mcap - eth_mcap
        
        # Calcular dominancias
        btc_dom = btc_mcap / total_mcap
        eth_dom = eth_mcap / total_mcap
        alt_dom = alt_mcap / total_mcap
    else:
        # Si no hay total_mcap, usar NaN (luego forward-fill)
        btc_dom = eth_dom = alt_dom = None
    
    dominancias.append({
        "btc_dominance": btc_dom,
        "eth_dominance": eth_dom,
        "alt_dominance": alt_dom
    })

# Añadir dominancias al DataFrame
df_dominancias = pd.DataFrame(dominancias, index=df_nuevo.index)
df_nuevo = df_nuevo.join(df_dominancias)

# Forward-fill para fechas sin total_market_cap
df_nuevo["btc_dominance"] = df_nuevo["btc_dominance"].ffill()
df_nuevo["eth_dominance"] = df_nuevo["eth_dominance"].ffill()
df_nuevo["alt_dominance"] = df_nuevo["alt_dominance"].ffill()

print(f"  ✓ Dominancias calculadas")
print(f"\n  📊 Estadísticas de dominancias:")
print(f"     BTC: {df_nuevo['btc_dominance'].iloc[-1]:.4f} ({df_nuevo['btc_dominance'].iloc[-1]*100:.2f}%)")
print(f"     ETH: {df_nuevo['eth_dominance'].iloc[-1]:.4f} ({df_nuevo['eth_dominance'].iloc[-1]*100:.2f}%)")
print(f"     ALT: {df_nuevo['alt_dominance'].iloc[-1]:.4f} ({df_nuevo['alt_dominance'].iloc[-1]*100:.2f}%)")

# 4. Añadir Fear & Greed
df_nuevo = df_nuevo.join(df_feargreed, how="left")
df_nuevo["fear_greed"] = df_nuevo["fear_greed"].ffill()
df_nuevo["FearGreed_Label"] = df_nuevo["FearGreed_Label"].ffill()
print(f"\n  ✓ Fear & Greed añadido")

# 5. Añadir inflation y fed_rate (forward-fill del último valor)
df_nuevo["inflation"] = df_merged["inflation"].iloc[-1]
df_nuevo["fed_rate"] = df_merged["fed_rate"].iloc[-1]
print(f"  ✓ Inflation y Fed Rate (forward-filled)")

# 6. Renombrar market_cap a mcap
df_nuevo.rename(columns={
    "btc_market_cap": "btc_mcap",
    "eth_market_cap": "eth_mcap"
}, inplace=True)

# 7. Drop total_market_cap (no está en df_merged)
df_nuevo.drop(columns=["total_market_cap", "total_volume"], inplace=True, errors="ignore")

# 8. Reordenar columnas para match con df_merged
columnas_orden = [
    "btc_open", "btc_high", "btc_low", "btc_close", "btc_volume",
    "eth_open", "eth_high", "eth_low", "eth_close", "eth_volume",
    "btc_dominance", "eth_dominance", "alt_dominance",
    "fear_greed", "FearGreed_Label",
    "inflation",
    "btc_mcap", "eth_mcap",
    "fed_rate"
]

df_nuevo = df_nuevo[columnas_orden]

print(f"\n✓ DataFrame nuevo formateado")
print(f"  Columnas: {len(df_nuevo.columns)} (match con df_merged)")
print(f"  NaN totales: {df_nuevo.isna().sum().sum()}")

df_nuevo.head()

🔧 Combinando datos y calculando dominancias...

  ✓ BTC + ETH combinados: 180 registros

  🧮 Calculando dominancias día a día...
  ✓ Dominancias calculadas

  📊 Estadísticas de dominancias:
     BTC: 0.5873 (58.73%)
     ETH: 0.1043 (10.43%)
     ALT: 0.3085 (30.85%)

  ✓ Fear & Greed añadido
  ✓ Inflation y Fed Rate (forward-filled)

✓ DataFrame nuevo formateado
  Columnas: 19 (match con df_merged)
  NaN totales: 0


,btc_open,btc_high,btc_low,btc_close,btc_volume,eth_open,eth_high,eth_low,eth_close,eth_volume,btc_dominance,eth_dominance,alt_dominance,fear_greed,FearGreed_Label,inflation,btc_mcap,eth_mcap,fed_rate
date,,,,,,,,,,,,,,,,,,,
2025-11-04,106521.086738,106521.086738,106521.086738,106521.086738,7.506843e+10,3600.715502,3600.715502,3600.715502,3600.715502,4.791406e+10,0.586718,0.119945,0.293337,21,Extreme Fear,2.7,2.123664e+12,4.341497e+11,3.84
2025-11-05,101635.274023,101635.274023,101635.274023,101635.274023,1.089657e+11,3296.743760,3296.743760,3296.743760,3296.743760,6.878429e+10,0.586756,0.115113,0.298132,23,Extreme Fear,2.7,2.026091e+12,3.974893e+11,3.84
2025-11-06,103877.959660,103877.959660,103877.959660,103877.959660,7.708391e+10,3427.690591,3427.690591,3427.690591,3427.690591,4.437453e+10,0.583813,0.116571,0.299616,27,Fear,2.7,2.073001e+12,4.139187e+11,3.84
2025-11-07,101322.640295,101322.640295,101322.640295,101322.640295,6.411822e+10,3308.918165,3308.918165,3308.918165,3308.918165,3.388097e+10,0.583309,0.115269,0.301423,24,Extreme Fear,2.7,2.019148e+12,3.990074e+11,3.84
2025-11-08,103396.084216,103396.084216,103396.084216,103396.084216,9.362700e+10,3434.351229,3434.351229,3434.351229,3434.351229,3.897260e+10,0.577857,0.116156,0.305987,20,Extreme Fear,2.7,2.063022e+12,4.146914e+11,3.84


In [32]:
print("🔧 Filtrando solo días nuevos...")

df_nuevo_filtrado = df_nuevo[df_nuevo.index > ultima_fecha].copy()

print(f"  ✓ Días nuevos: {len(df_nuevo_filtrado)}")
print(f"  📅 Rango: {df_nuevo_filtrado.index.min().date()} → {df_nuevo_filtrado.index.max().date()}")

if df_nuevo_filtrado.empty:
    print("⚠️  No hay días nuevos tras filtrar")
    raise SystemExit("Sin días nuevos")

df_nuevo_filtrado.tail()

🔧 Filtrando solo días nuevos...
  ✓ Días nuevos: 128
  📅 Rango: 2025-12-26 → 2026-05-02


,btc_open,btc_high,btc_low,btc_close,btc_volume,eth_open,eth_high,eth_low,eth_close,eth_volume,btc_dominance,eth_dominance,alt_dominance,fear_greed,FearGreed_Label,inflation,btc_mcap,eth_mcap,fed_rate
date,,,,,,,,,,,,,,,,,,,
2026-04-28,77361.299110,77361.299110,77361.299110,77361.299110,3.945377e+10,2299.770459,2299.770459,2299.770459,2299.770459,1.215075e+10,0.582147,0.104248,0.313605,33,Fear,2.7,1.549703e+12,2.775134e+11,3.84
2026-04-29,76345.225210,76345.225210,76345.225210,76345.225210,3.268317e+10,2288.044927,2288.044927,2288.044927,2288.044927,1.206684e+10,0.579621,0.104719,0.315660,26,Fear,2.7,1.528431e+12,2.761383e+11,3.84
2026-04-30,75774.885523,75774.885523,75774.885523,75774.885523,4.252829e+10,2253.458358,2253.458358,2253.458358,2253.458358,1.937937e+10,0.580154,0.104013,0.315834,29,Fear,2.7,1.517195e+12,2.720097e+11,3.84
2026-05-01,76286.577110,76286.577110,76286.577110,76286.577110,3.171787e+10,2255.980292,2255.980292,2255.980292,2255.980292,1.136939e+10,0.581572,0.103643,0.314785,26,Fear,2.7,1.528164e+12,2.723357e+11,3.84
2026-05-02,78473.604006,78473.604006,78473.604006,78473.604006,1.897858e+10,2311.336101,2311.336101,2311.336101,2311.336101,6.081074e+09,0.587278,0.104261,0.308461,39,Fear,2.7,1.571453e+12,2.789837e+11,3.84


In [35]:
print("🔗 Mergeando con df_merged...")

# Concatenar
df_actualizado = pd.concat([df_merged, df_nuevo_filtrado])

# Eliminar duplicados (keep last por si actualizaste algo)
df_actualizado = df_actualizado[~df_actualizado.index.duplicated(keep="last")]

# Ordenar por fecha
df_actualizado.sort_index(inplace=True)

print(f"  ✓ Merge completado")
print(f"  📊 Total registros: {len(df_actualizado)} ({len(df_merged)} antiguos + {len(df_nuevo_filtrado)} nuevos)")
print(f"  📅 Última fecha: {df_actualizado.index.max().date()}")

df_actualizado.tail()

🔗 Mergeando con df_merged...
  ✓ Merge completado
  📊 Total registros: 3013 (2885 antiguos + 128 nuevos)
  📅 Última fecha: 2026-05-02


,btc_open,btc_high,btc_low,btc_close,btc_volume,eth_open,eth_high,eth_low,eth_close,eth_volume,btc_dominance,eth_dominance,alt_dominance,fear_greed,FearGreed_Label,inflation,btc_mcap,eth_mcap,fed_rate
date,,,,,,,,,,,,,,,,,,,
2026-04-28,77361.299110,77361.299110,77361.299110,77361.299110,3.945377e+10,2299.770459,2299.770459,2299.770459,2299.770459,1.215075e+10,0.582147,0.104248,0.313605,33.0,Fear,2.7,1.549703e+12,2.775134e+11,3.84
2026-04-29,76345.225210,76345.225210,76345.225210,76345.225210,3.268317e+10,2288.044927,2288.044927,2288.044927,2288.044927,1.206684e+10,0.579621,0.104719,0.315660,26.0,Fear,2.7,1.528431e+12,2.761383e+11,3.84
2026-04-30,75774.885523,75774.885523,75774.885523,75774.885523,4.252829e+10,2253.458358,2253.458358,2253.458358,2253.458358,1.937937e+10,0.580154,0.104013,0.315834,29.0,Fear,2.7,1.517195e+12,2.720097e+11,3.84
2026-05-01,76286.577110,76286.577110,76286.577110,76286.577110,3.171787e+10,2255.980292,2255.980292,2255.980292,2255.980292,1.136939e+10,0.581572,0.103643,0.314785,26.0,Fear,2.7,1.528164e+12,2.723357e+11,3.84
2026-05-02,78473.604006,78473.604006,78473.604006,78473.604006,1.897858e+10,2311.336101,2311.336101,2311.336101,2311.336101,6.081074e+09,0.587278,0.104261,0.308461,39.0,Fear,2.7,1.571453e+12,2.789837e+11,3.84


In [36]:
print("💾 Guardando CSV actualizado...")

# Backup del CSV antiguo
import shutil
backup_path = RUTA_DF_MERGED.replace(".csv", f"_backup_{hoy.strftime('%Y%m%d')}.csv")
shutil.copy(RUTA_DF_MERGED, backup_path)
print(f"  ✓ Backup guardado: {backup_path}")

# Guardar CSV actualizado
df_actualizado.to_csv(RUTA_DF_MERGED)
print(f"  ✓ CSV actualizado guardado: {RUTA_DF_MERGED}")
print(f"\n📊 Resumen final:")
print(f"   Total registros: {len(df_actualizado)}")
print(f"   Rango: {df_actualizado.index.min().date()} → {df_actualizado.index.max().date()}")
print(f"   Días añadidos: {len(df_nuevo_filtrado)}")

💾 Guardando CSV actualizado...
  ✓ Backup guardado: C:\Users\josit\CUARTO CURSO\TFG\TFG_Crypto_DEF\data\csv\df_merged_backup_20260502.csv
  ✓ CSV actualizado guardado: C:\Users\josit\CUARTO CURSO\TFG\TFG_Crypto_DEF\data\csv\df_merged.csv

📊 Resumen final:
   Total registros: 3013
   Rango: 2018-02-01 → 2026-05-02
   Días añadidos: 128
